# Model Serving metrics setup

This notebook configures a continuous serverless collector and publishes an AI/BI dashboard for Model Serving endpoint metrics. It discovers all non-foundation workspace endpoints when the include list is empty and adds inference-table datasets when payload logging is already available.

Edit the configuration cell below, then run the notebook from top to bottom.

In [ ]:
# Run the entire notebook after editing this cell.
# Inference tables are detected but never enabled by this setup.

# Optionally restrict collection to these endpoint names. Leave empty to discover all non-foundation endpoints.
INCLUDE_ENDPOINT_NAMES: list[str] = [
    # Add endpoints here
]

# Unity Catalog catalog used for collected metrics.
STORAGE_CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# Unity Catalog schema created for collected metrics and dashboard views.
STORAGE_SCHEMA = "model_serving_observability"

# Name of the recurring Lakeflow Job.
JOB_NAME = "Model serving metrics sync"

# Name of the AI/BI dashboard.
DASHBOARD_NAME = "Model Serving metrics"

# Workspace notebook name used by the recurring collection task.
COLLECTOR_NOTEBOOK_NAME = "model_serving_metrics_collector"

# Seconds between endpoint metric scrapes in the continuous collector.
COLLECTION_INTERVAL_SECONDS = 60

# PAUSED or UNPAUSED continuous job state.
COLLECTION_PAUSE_STATUS = "UNPAUSED"

# SQL warehouse ID for the dashboard, or empty to use the first available warehouse.
WAREHOUSE_ID = ""

# Workspace folder for the dashboard, or empty to use this notebook's folder.
PARENT_PATH = ""

# Serverless performance target. STANDARD minimizes cost and startup optimization overhead.
COLLECTOR_PERFORMANCE_TARGET = "STANDARD"

# Job environment key used by the collector task.
COLLECTOR_ENVIRONMENT_KEY = "collector"

# Databricks serverless environment version used by the collector task.
COLLECTOR_ENVIRONMENT_VERSION = "4"

# Python dependencies installed in the collector's serverless environment.
COLLECTOR_DEPENDENCIES = [
    "prometheus-client>=0.20,<1",
    "databricks-sdk>=0.36.0",
]

# Rolling dashboard time window in hours.
DASHBOARD_LOOKBACK_HOURS = 24

## Discover workspace resources

Resolve endpoint names, validate continuous collection settings, detect inference tables, select a SQL warehouse, and derive workspace paths.

In [ ]:
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import quote

from databricks.sdk import WorkspaceClient


def _print_log(event: str, **fields: Any) -> None:
    """Write one structured status line to notebook output."""
    print(
        json.dumps(
            {
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "event": event,
                **fields,
            },
            default=str,
            sort_keys=True,
        ),
        flush=True,
    )


workspace = WorkspaceClient()
WORKSPACE_HOST = workspace.config.host.rstrip("/")
_print_log(
    "setup_configuration",
    include_endpoint_names=INCLUDE_ENDPOINT_NAMES,
    storage_catalog=STORAGE_CATALOG,
    storage_schema=STORAGE_SCHEMA,
    job_name=JOB_NAME,
    dashboard_name=DASHBOARD_NAME,
    collector_notebook_name=COLLECTOR_NOTEBOOK_NAME,
    collection_interval_seconds=COLLECTION_INTERVAL_SECONDS,
    collection_pause_status=COLLECTION_PAUSE_STATUS,
    warehouse_id=WAREHOUSE_ID or None,
    parent_path=PARENT_PATH or None,
    collector_performance_target=COLLECTOR_PERFORMANCE_TARGET,
    collector_environment_key=COLLECTOR_ENVIRONMENT_KEY,
    collector_environment_version=COLLECTOR_ENVIRONMENT_VERSION,
    collector_dependencies=COLLECTOR_DEPENDENCIES,
    dashboard_lookback_hours=DASHBOARD_LOOKBACK_HOURS,
    workspace_host=WORKSPACE_HOST,
)


def _is_foundation_endpoint(endpoint: Any) -> bool:
    """Return whether an endpoint routes to a system.ai foundation model."""
    endpoint_dict = endpoint.as_dict()
    config = endpoint_dict.get("config") or {}
    entities = config.get("served_entities") or config.get("served_models") or []
    return any(
        str(entity.get("entity_name") or entity.get("model_name") or "").startswith("system.ai.")
        for entity in entities
    )


def _resolve_endpoint_names(configured_names: str | list[str] | None) -> list[str]:
    """Validate an optional include list or discover all non-foundation endpoints."""
    if isinstance(configured_names, list):
        names = [name.strip() for name in configured_names if name.strip()]
    else:
        text = (configured_names or "").strip()
        if text.startswith("["):
            parsed = json.loads(text)
            if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):
                raise ValueError("JSON INCLUDE_ENDPOINT_NAMES must be an array of strings")
            names = [name.strip() for name in parsed if name.strip()]
        else:
            names = [name.strip() for name in text.split(",") if name.strip()]

    endpoints = [endpoint for endpoint in workspace.serving_endpoints.list() if endpoint.name]
    endpoint_by_name = {endpoint.name: endpoint for endpoint in endpoints}
    foundation_names = {
        endpoint.name for endpoint in endpoints if _is_foundation_endpoint(endpoint)
    }
    if names:
        requested_names = set(names)
        missing_names = sorted(requested_names - endpoint_by_name.keys())
        if missing_names:
            raise ValueError(f"Included serving endpoints were not found: {missing_names}")
        blocked_names = sorted(requested_names & foundation_names)
        if blocked_names:
            raise ValueError(
                "Foundation Model API endpoints cannot be collected by export-metrics: "
                f"{blocked_names}"
            )
        resolved_names = sorted(requested_names)
        _print_log("endpoint_include_resolved", endpoints=resolved_names)
        return resolved_names

    resolved_names = sorted(endpoint_by_name.keys() - foundation_names)
    _print_log(
        "endpoint_discovery_complete",
        endpoint_count=len(resolved_names),
        endpoints=resolved_names,
        skipped_foundation_count=len(foundation_names),
        skipped_foundation_endpoints=sorted(foundation_names),
    )
    return resolved_names


def _configured_inference_table(endpoint_name: str) -> str | None:
    """Read one endpoint's API configuration and derive its payload table name."""
    endpoint = workspace.serving_endpoints.get(endpoint_name).as_dict()
    gateway_config = (endpoint.get("ai_gateway") or {}).get("inference_table_config") or {}
    if gateway_config.get("enabled"):
        catalog_name = gateway_config.get("catalog_name")
        schema_name = gateway_config.get("schema_name")
        table_prefix = gateway_config.get("table_name_prefix")
        if catalog_name and schema_name and table_prefix:
            return f"{catalog_name}.{schema_name}.{table_prefix}_payload"

    legacy_config = (endpoint.get("config") or {}).get("auto_capture_config") or {}
    if not legacy_config.get("enabled"):
        return None

    payload_table = ((legacy_config.get("state") or {}).get("payload_table") or {}).get("name")
    if payload_table:
        return payload_table

    catalog_name = legacy_config.get("catalog_name")
    schema_name = legacy_config.get("schema_name")
    table_prefix = legacy_config.get("table_name_prefix")
    if catalog_name and schema_name and table_prefix:
        return f"{catalog_name}.{schema_name}.{table_prefix}_payload"
    return None


def _validate_metrics_export(endpoint_name: str) -> None:
    """Fail when an explicitly included endpoint cannot export metrics."""
    response = workspace.api_client.do(
        "GET",
        f"/api/2.0/serving-endpoints/{quote(endpoint_name, safe='')}/metrics",
        headers={"Accept": "text/plain"},
        raw=True,
    )
    contents = response.get("contents") if isinstance(response, dict) else None
    payload = contents.read() if hasattr(contents, "read") else contents
    if not payload:
        raise RuntimeError(f"Metrics export returned no content for endpoint {endpoint_name!r}")
    _print_log(
        "endpoint_metrics_export_validated",
        endpoint_name=endpoint_name,
        response_bytes=len(payload),
    )


def _sql_identifier(identifier: str) -> str:
    """Quote a three-part Unity Catalog identifier."""
    return ".".join(f"`{part.replace('`', '``')}`" for part in identifier.split("."))


def _sql_literal(value: str) -> str:
    """Quote a SQL string literal."""
    return "'" + value.replace("'", "''") + "'"


for setting_name, setting_value in {
    "JOB_NAME": JOB_NAME,
    "DASHBOARD_NAME": DASHBOARD_NAME,
    "COLLECTOR_NOTEBOOK_NAME": COLLECTOR_NOTEBOOK_NAME,
    "COLLECTOR_ENVIRONMENT_KEY": COLLECTOR_ENVIRONMENT_KEY,
    "COLLECTOR_ENVIRONMENT_VERSION": COLLECTOR_ENVIRONMENT_VERSION,
}.items():
    if not setting_value.strip():
        raise ValueError(f"{setting_name} cannot be empty")
if not COLLECTOR_DEPENDENCIES:
    raise ValueError("COLLECTOR_DEPENDENCIES must contain collector runtime dependencies")

identifier_pattern = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
for identifier_name, identifier_value in {
    "STORAGE_CATALOG": STORAGE_CATALOG,
    "STORAGE_SCHEMA": STORAGE_SCHEMA,
}.items():
    if not identifier_pattern.fullmatch(identifier_value):
        raise ValueError(
            f"{identifier_name} must contain only letters, numbers, and underscores: "
            f"{identifier_value!r}"
        )

if COLLECTION_PAUSE_STATUS not in {"PAUSED", "UNPAUSED"}:
    raise ValueError("COLLECTION_PAUSE_STATUS must be PAUSED or UNPAUSED")
if COLLECTOR_PERFORMANCE_TARGET not in {"STANDARD", "PERFORMANCE_OPTIMIZED"}:
    raise ValueError("COLLECTOR_PERFORMANCE_TARGET must be STANDARD or PERFORMANCE_OPTIMIZED")
if COLLECTION_INTERVAL_SECONDS < 10:
    raise ValueError("COLLECTION_INTERVAL_SECONDS must be at least 10")
if DASHBOARD_LOOKBACK_HOURS < 1:
    raise ValueError("DASHBOARD_LOOKBACK_HOURS must be at least 1")
available_catalogs = {row[0] for row in spark.sql("SHOW CATALOGS").collect()}
if STORAGE_CATALOG not in available_catalogs:
    raise RuntimeError(
        f"Unity Catalog catalog was not found: {STORAGE_CATALOG}. "
        f"Available catalogs: {sorted(available_catalogs)}"
    )
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {STORAGE_CATALOG}.{STORAGE_SCHEMA}")
_print_log(
    "unity_catalog_destination_validated",
    catalog=STORAGE_CATALOG,
    schema=STORAGE_SCHEMA,
)

RESOLVED_ENDPOINT_NAMES = _resolve_endpoint_names(INCLUDE_ENDPOINT_NAMES)
if not RESOLVED_ENDPOINT_NAMES:
    raise RuntimeError("The workspace contains no collectable Model Serving endpoints")
if INCLUDE_ENDPOINT_NAMES:
    for endpoint_name in RESOLVED_ENDPOINT_NAMES:
        _validate_metrics_export(endpoint_name)

CONFIGURED_INFERENCE_TABLES = {
    endpoint_name: table_name
    for endpoint_name in RESOLVED_ENDPOINT_NAMES
    if (table_name := _configured_inference_table(endpoint_name)) is not None
}
AVAILABLE_INFERENCE_TABLES = {
    endpoint_name: table_name
    for endpoint_name, table_name in CONFIGURED_INFERENCE_TABLES.items()
    if spark.catalog.tableExists(table_name)
}
if AVAILABLE_INFERENCE_TABLES:
    INFERENCE_TABLE_STATUS = "available"
elif CONFIGURED_INFERENCE_TABLES:
    INFERENCE_TABLE_STATUS = "configured"
else:
    INFERENCE_TABLE_STATUS = "unavailable"
missing_inference_tables = {
    endpoint_name: table_name
    for endpoint_name, table_name in CONFIGURED_INFERENCE_TABLES.items()
    if endpoint_name not in AVAILABLE_INFERENCE_TABLES
}
if missing_inference_tables:
    _print_log(
        "inference_tables_missing",
        level="warning",
        tables=missing_inference_tables,
    )
_print_log(
    "inference_table_discovery_complete",
    status=INFERENCE_TABLE_STATUS,
    configured_tables=CONFIGURED_INFERENCE_TABLES,
    available_tables=AVAILABLE_INFERENCE_TABLES,
)

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
resolved_parent_path = PARENT_PATH.strip().rstrip("/") or str(Path(notebook_path).parent)
collector_notebook_path = str(Path(notebook_path).with_name(COLLECTOR_NOTEBOOK_NAME))
if not collector_notebook_path.startswith("/"):
    collector_notebook_path = f"/{collector_notebook_path.lstrip('/')}"
collector_status = workspace.api_client.do(
    "GET",
    "/api/2.0/workspace/get-status",
    query={"path": collector_notebook_path},
)
if collector_status.get("object_type") != "NOTEBOOK":
    raise RuntimeError(
        f"Collector notebook was not found at {collector_notebook_path}: {collector_status}"
    )
parent_status = workspace.api_client.do(
    "GET",
    "/api/2.0/workspace/get-status",
    query={"path": resolved_parent_path},
)
if parent_status.get("object_type") != "DIRECTORY":
    raise RuntimeError(
        f"Dashboard parent path is not a workspace directory: {resolved_parent_path}"
    )
_print_log(
    "workspace_paths_validated",
    setup_notebook_path=notebook_path,
    collector_notebook_path=collector_notebook_path,
    dashboard_parent_path=resolved_parent_path,
)

if WAREHOUSE_ID:
    resolved_warehouse_id = WAREHOUSE_ID
else:
    warehouses = [warehouse for warehouse in workspace.warehouses.list() if warehouse.id]
    if not warehouses:
        raise RuntimeError("No SQL warehouse is available. Set WAREHOUSE_ID in the first cell.")
    warehouses.sort(
        key=lambda warehouse: (
            str(getattr(warehouse, "state", "")) != "State.RUNNING",
            not bool(getattr(warehouse, "enable_serverless_compute", False)),
            str(getattr(warehouse, "name", "")),
        )
    )
    resolved_warehouse_id = warehouses[0].id
resolved_warehouse = workspace.warehouses.get(resolved_warehouse_id)
_print_log(
    "sql_warehouse_validated",
    warehouse_id=resolved_warehouse_id,
    warehouse_name=resolved_warehouse.name,
    warehouse_state=str(resolved_warehouse.state),
    serverless=resolved_warehouse.enable_serverless_compute,
)

TABLE_PREFIX = f"{STORAGE_CATALOG}.{STORAGE_SCHEMA}"
ENDPOINT_SNAPSHOTS_TABLE = f"{TABLE_PREFIX}.model_serving_endpoint_snapshots"
RAW_METRICS_TABLE = f"{TABLE_PREFIX}.model_serving_metrics_raw"
METRIC_SAMPLES_TABLE = f"{TABLE_PREFIX}.model_serving_metric_samples"
COUNTER_RATES_VIEW = f"{TABLE_PREFIX}.model_serving_counter_rates"
HISTOGRAM_PERCENTILES_VIEW = f"{TABLE_PREFIX}.model_serving_histogram_percentiles"
DASHBOARD_SERIES_VIEW = f"{TABLE_PREFIX}.model_serving_dashboard_series"

## Configure the continuous collector

Create or update the continuous serverless Lakeflow Job and wait for its Delta views when they do not exist yet.

In [ ]:
collector_parameters = {
    "include_endpoint_names": json.dumps(INCLUDE_ENDPOINT_NAMES),
    "storage_catalog": STORAGE_CATALOG,
    "storage_schema": STORAGE_SCHEMA,
    "poll_interval_seconds": str(COLLECTION_INTERVAL_SECONDS),
}
job_settings = {
    "name": JOB_NAME,
    "description": (
        "Continuously scrapes Model Serving Prometheus metrics into Delta and discovers "
        "configured inference tables from each endpoint API."
    ),
    "tags": {"inference_tables": INFERENCE_TABLE_STATUS},
    "performance_target": COLLECTOR_PERFORMANCE_TARGET,
    "max_concurrent_runs": 1,
    "continuous": {
        "pause_status": COLLECTION_PAUSE_STATUS,
        "task_retry_mode": "ON_FAILURE",
    },
    "parameters": [
        {"name": name, "default": value}
        for name, value in collector_parameters.items()
    ],
    "environments": [
        {
            "environment_key": COLLECTOR_ENVIRONMENT_KEY,
            "spec": {
                "environment_version": COLLECTOR_ENVIRONMENT_VERSION,
                "dependencies": COLLECTOR_DEPENDENCIES,
            },
        }
    ],
    "tasks": [
        {
            "task_key": "scrape_serving_metrics",
            "environment_key": COLLECTOR_ENVIRONMENT_KEY,
            "notebook_task": {
                "notebook_path": collector_notebook_path,
                "source": "WORKSPACE",
                "base_parameters": collector_parameters,
            },
        }
    ],
}

matching_jobs = list(workspace.jobs.list(name=JOB_NAME))
if len(matching_jobs) > 1:
    raise RuntimeError(
        f"Multiple jobs named {JOB_NAME!r} exist: "
        f"{[job.job_id for job in matching_jobs]}"
    )
_print_log(
    "job_configuration_resolved",
    job_name=JOB_NAME,
    existing_job_id=matching_jobs[0].job_id if matching_jobs else None,
    collector_parameters=collector_parameters,
    continuous=job_settings["continuous"],
    environment=job_settings["environments"][0],
    performance_target=COLLECTOR_PERFORMANCE_TARGET,
)
if matching_jobs:
    existing_job = matching_jobs[0]
    if not existing_job.job_id:
        raise RuntimeError(f"Existing job {JOB_NAME!r} has no job ID")
    workspace.api_client.do(
        "POST",
        "/api/2.1/jobs/reset",
        body={"job_id": existing_job.job_id, "new_settings": job_settings},
    )
    job_id = existing_job.job_id
    job_action = "updated"
else:
    created_job = workspace.api_client.do("POST", "/api/2.1/jobs/create", body=job_settings)
    job_id = created_job.get("job_id")
    if not job_id:
        raise RuntimeError(f"Job create response did not contain job_id: {created_job}")
    job_action = "created"
_print_log("job_configuration_applied", action=job_action, job_id=job_id)

def _latest_continuous_run() -> dict[str, Any] | None:
    response = workspace.api_client.do(
        "GET",
        "/api/2.1/jobs/runs/list",
        query={"job_id": job_id, "limit": 10},
    )
    return next(
        (
            run
            for run in response.get("runs", [])
            if run.get("trigger") == "CONTINUOUS"
        ),
        None,
    )


if COLLECTION_PAUSE_STATUS == "PAUSED":
    if not spark.catalog.tableExists(DASHBOARD_SERIES_VIEW):
        raise RuntimeError(
            "The collector is PAUSED and dashboard views do not exist. "
            "Set COLLECTION_PAUSE_STATUS to UNPAUSED for initial setup."
        )
    _print_log("collector_paused", job_id=job_id)
else:
    for attempt in range(1, 31):
        continuous_run = _latest_continuous_run()
        run_state = (continuous_run or {}).get("state") or {}
        life_cycle_state = run_state.get("life_cycle_state", "PENDING")
        result_state = run_state.get("result_state")
        state_message = run_state.get("state_message", "")
        views_ready = spark.catalog.tableExists(DASHBOARD_SERIES_VIEW)
        _print_log(
            "collector_startup_poll",
            attempt=attempt,
            job_id=job_id,
            run_id=(continuous_run or {}).get("run_id"),
            life_cycle_state=life_cycle_state,
            result_state=result_state,
            state_message=state_message,
            views_ready=views_ready,
        )
        if life_cycle_state in {"INTERNAL_ERROR", "SKIPPED"} or (
            life_cycle_state == "TERMINATED" and result_state != "SUCCESS"
        ):
            raise RuntimeError(
                f"Continuous collector failed during startup: state={life_cycle_state} "
                f"result={result_state} message={state_message}"
            )
        if life_cycle_state == "RUNNING" and views_ready:
            break
        time.sleep(10)
    else:
        raise TimeoutError(
            f"Continuous collector job {job_id} was not running with initialized views "
            "within 5 minutes"
        )

## Prepare dashboard datasets

Define reusable dashboard widgets, build the rolling metrics queries, and validate every dataset query before publication.

In [ ]:
def _query_lines(sql: str) -> list[str]:
    """Split SQL into Lakeview query lines while preserving separators."""
    return [f"{line}\n" for line in sql.strip().splitlines()]


def _text_widget(name: str, text: str, x: int, y: int, width: int, height: int) -> dict:
    return {
        "widget": {"name": name, "multilineTextboxSpec": {"lines": [text]}},
        "position": {"x": x, "y": y, "width": width, "height": height},
    }


def _counter_widget(
    dataset_name: str,
    time_column: str,
    name: str,
    title: str,
    field_name: str,
    expression: str,
    x: int,
    y: int,
    format_spec: dict[str, Any] | None = None,
) -> dict:
    value_encoding: dict[str, Any] = {"fieldName": field_name, "displayName": title}
    if format_spec:
        value_encoding["format"] = format_spec
    period_name = f"hourly({time_column})"
    return {
        "widget": {
            "name": name,
            "queries": [
                {
                    "name": "main_query",
                    "query": {
                        "datasetName": dataset_name,
                        "fields": [
                            {"name": field_name, "expression": expression},
                            {
                                "name": period_name,
                                "expression": f'DATE_TRUNC("HOUR", `{time_column}`)',
                            },
                        ],
                        "disaggregated": False,
                    },
                }
            ],
            "spec": {
                "version": 2,
                "widgetType": "counter",
                "encodings": {
                    "value": value_encoding,
                    "period": {"fieldName": period_name},
                },
                "frame": {"showTitle": True, "title": title},
            },
        },
        "position": {"x": x, "y": y, "width": 3, "height": 4},
    }


def _line_widget(
    dataset_name: str,
    time_column: str,
    name: str,
    title: str,
    y_fields: list[tuple[str, str, str]],
    x: int,
    y: int,
    format_spec: dict[str, Any] | None = None,
) -> dict:
    fields = [{"name": time_column, "expression": f"`{time_column}`"}]
    y_encodings: list[dict[str, Any]] = []
    for field_name, expression, display_name in y_fields:
        fields.append({"name": field_name, "expression": expression})
        encoding: dict[str, Any] = {"fieldName": field_name, "displayName": display_name}
        if format_spec:
            encoding["format"] = format_spec
        y_encodings.append(encoding)
    return {
        "widget": {
            "name": name,
            "queries": [
                {
                    "name": "main_query",
                    "query": {
                        "datasetName": dataset_name,
                        "fields": fields,
                        "disaggregated": False,
                    },
                }
            ],
            "spec": {
                "version": 3,
                "widgetType": "line",
                "encodings": {
                    "x": {
                        "fieldName": time_column,
                        "scale": {"type": "temporal"},
                        "displayName": "Timestamp",
                    },
                    "y": {"scale": {"type": "quantitative"}, "fields": y_encodings},
                },
                "frame": {"showTitle": True, "title": title},
            },
        },
        "position": {"x": x, "y": y, "width": 6, "height": 6},
    }


def _filter_widget(
    dataset_name: str,
    field_name: str,
    title: str,
    name: str,
    widget_type: str,
    x: int,
    y: int,
    width: int,
) -> dict:
    query_name = f"{dataset_name}_{field_name}"
    return {
        "widget": {
            "name": name,
            "queries": [
                {
                    "name": query_name,
                    "query": {
                        "datasetName": dataset_name,
                        "fields": [{"name": field_name, "expression": f"`{field_name}`"}],
                        "disaggregated": False,
                    },
                }
            ],
            "spec": {
                "version": 2,
                "widgetType": widget_type,
                "encodings": {
                    "fields": [
                        {
                            "fieldName": field_name,
                            "displayName": title,
                            "queryName": query_name,
                        }
                    ]
                },
                "frame": {"showTitle": True, "title": title},
            },
        },
        "position": {"x": x, "y": y, "width": width, "height": 2},
    }


def _inference_query() -> str | None:
    """Build request-level aggregates for every available inference table."""
    selects: list[str] = []
    for endpoint_name, table_name in AVAILABLE_INFERENCE_TABLES.items():
        columns = set(spark.table(table_name).columns)
        time_column = next(
            (column for column in ("event_time", "request_time", "timestamp") if column in columns),
            None,
        )
        latency_column = next(
            (
                column
                for column in ("latency_ms", "execution_duration_ms", "execution_time_ms")
                if column in columns
            ),
            None,
        )
        if not time_column or "status_code" not in columns:
            _print_log(
                "inference_table_schema_unsupported",
                level="warning",
                endpoint_name=endpoint_name,
                table_name=table_name,
                columns=sorted(columns),
            )
            continue

        latency_expression = (
            f"percentile_approx(CAST(`{latency_column}` AS DOUBLE), 0.50)"
            if latency_column
            else "CAST(NULL AS DOUBLE)"
        )
        latency_p99_expression = (
            f"percentile_approx(CAST(`{latency_column}` AS DOUBLE), 0.99)"
            if latency_column
            else "CAST(NULL AS DOUBLE)"
        )
        ttfb_expression = (
            "percentile_approx(CAST(`time_to_first_byte_ms` AS DOUBLE), 0.99)"
            if "time_to_first_byte_ms" in columns
            else "CAST(NULL AS DOUBLE)"
        )
        selects.append(
            f"""
SELECT
  DATE_TRUNC('MINUTE', `{time_column}`) AS observed_at,
  {_sql_literal(endpoint_name)} AS endpoint_name,
  COUNT(*) AS request_count,
  SUM(CASE WHEN status_code >= 400 THEN 1 ELSE 0 END) AS error_count,
  {latency_expression} AS latency_p50,
  {latency_p99_expression} AS latency_p99,
  {ttfb_expression} AS time_to_first_byte_p99
FROM {_sql_identifier(table_name)}
WHERE `{time_column}` >= timestampadd(HOUR, -{DASHBOARD_LOOKBACK_HOURS}, current_timestamp())
GROUP BY DATE_TRUNC('MINUTE', `{time_column}`)
""".strip()
        )
    return "\nUNION ALL\n".join(selects) if selects else None


metrics_query = f"""
SELECT
  scraped_at,
  endpoint_name,
  MAX(CASE WHEN chart_name = 'Response rate (per second)' THEN value END) AS response_rate,
  MAX(CASE WHEN chart_name = 'Response error rates (per second)' AND series_name = '4XX errors per second' THEN value END) AS error_4xx_rate,
  MAX(CASE WHEN chart_name = 'Response error rates (per second)' AND series_name = '5XX errors per second' THEN value END) AS error_5xx_rate,
  MAX(CASE WHEN chart_name = 'Latency (ms)' AND series_name = 'p50' THEN value END) AS latency_p50,
  MAX(CASE WHEN chart_name = 'Latency (ms)' AND series_name = 'p99' THEN value END) AS latency_p99,
  MAX(CASE WHEN chart_name = 'Model inference latency (ms)' AND series_name = 'p50' THEN value END) AS inference_p50,
  MAX(CASE WHEN chart_name = 'Model inference latency (ms)' AND series_name = 'p99' THEN value END) AS inference_p99,
  MAX(CASE WHEN chart_name = 'Request queue wait time (ms)' AND series_name = 'p50' THEN value END) AS queue_p50,
  MAX(CASE WHEN chart_name = 'Request queue wait time (ms)' AND series_name = 'p99' THEN value END) AS queue_p99,
  MAX(CASE WHEN chart_name = 'CPU Usage (%)' THEN value END) AS cpu_pct,
  MAX(CASE WHEN chart_name = 'Memory Usage (%)' THEN value END) AS mem_pct,
  MAX(CASE WHEN chart_name = 'Provisioned Concurrency' THEN value END) AS provisioned_concurrency
FROM model_serving_dashboard_series
WHERE chart_name IS NOT NULL
  AND scraped_at >= timestampadd(HOUR, -{DASHBOARD_LOOKBACK_HOURS}, current_timestamp())
GROUP BY scraped_at, endpoint_name
""".strip()

metrics_validation_query = metrics_query.replace(
    "FROM model_serving_dashboard_series",
    f"FROM {_sql_identifier(DASHBOARD_SERIES_VIEW)}",
)
metrics_validation_rows = spark.sql(metrics_validation_query).limit(1).collect()
_print_log(
    "dashboard_metrics_query_validated",
    returned_sample_row=bool(metrics_validation_rows),
    source=DASHBOARD_SERIES_VIEW,
)
inference_query = _inference_query()
if inference_query:
    inference_validation_rows = spark.sql(inference_query).limit(1).collect()
    _print_log(
        "dashboard_inference_query_validated",
        returned_sample_row=bool(inference_validation_rows),
        source_tables=AVAILABLE_INFERENCE_TABLES,
    )
else:
    _print_log("dashboard_inference_query_skipped", reason="no_available_inference_tables")

## Build the dashboard definition

Assemble the overview page and add an inference-table page only when payload tables are configured and available.

In [ ]:
percent_format = {"type": "number", "decimalPlaces": {"type": "exact", "places": 1}}
rate_format = {"type": "number", "decimalPlaces": {"type": "max", "places": 2}}
ms_format = {"type": "number", "decimalPlaces": {"type": "max", "places": 1}}
count_format = {"type": "number", "abbreviation": "compact"}

datasets = [
    {
        "name": "ds_series",
        "displayName": "Serving metric series",
        "queryLines": _query_lines(metrics_query),
    }
]
overview_layout = [
    _text_widget("title", "## Model Serving metrics", 0, 0, 12, 1),
    _text_widget(
        "subtitle",
        f"Rolling {DASHBOARD_LOOKBACK_HOURS}-hour latency, request rates, errors, and replica usage.",
        0,
        1,
        12,
        1,
    ),
    _filter_widget(
        "ds_series", "endpoint_name", "Model endpoint", "filter_endpoint_name",
        "filter-single-select", 0, 2, 4,
    ),
    _filter_widget(
        "ds_series", "scraped_at", "Time range", "filter_scraped_at",
        "filter-date-range-picker", 4, 2, 8,
    ),
    _counter_widget(
        "ds_series", "scraped_at", "kpi_response_rate", "Response rate",
        "avg(response_rate)", "AVG(`response_rate`)", 0, 4, rate_format,
    ),
    _counter_widget(
        "ds_series", "scraped_at", "kpi_latency_p99", "Latency p99 (ms)",
        "avg(latency_p99)", "AVG(`latency_p99`)", 3, 4, ms_format,
    ),
    _counter_widget(
        "ds_series", "scraped_at", "kpi_error_5xx", "5XX errors / sec",
        "avg(error_5xx_rate)", "AVG(`error_5xx_rate`)", 6, 4, rate_format,
    ),
    _counter_widget(
        "ds_series", "scraped_at", "kpi_cpu", "CPU usage",
        "avg(cpu_pct)", "AVG(`cpu_pct`)", 9, 4, percent_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_latency", "Latency (ms)",
        [
            ("avg(latency_p50)", "AVG(`latency_p50`)", "p50"),
            ("avg(latency_p99)", "AVG(`latency_p99`)", "p99"),
        ],
        0, 8, ms_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_inference", "Model inference latency (ms)",
        [
            ("avg(inference_p50)", "AVG(`inference_p50`)", "p50"),
            ("avg(inference_p99)", "AVG(`inference_p99`)", "p99"),
        ],
        6, 8, ms_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_queue", "Request queue wait time (ms)",
        [
            ("avg(queue_p50)", "AVG(`queue_p50`)", "p50"),
            ("avg(queue_p99)", "AVG(`queue_p99`)", "p99"),
        ],
        0, 14, ms_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_response_rate", "Response rate (per second)",
        [("avg(response_rate)", "AVG(`response_rate`)", "response rate")],
        6, 14, rate_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_errors", "Response error rates (per second)",
        [
            ("avg(error_4xx_rate)", "AVG(`error_4xx_rate`)", "4XX errors per second"),
            ("avg(error_5xx_rate)", "AVG(`error_5xx_rate`)", "5XX errors per second"),
        ],
        0, 20, rate_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_cpu", "CPU Usage (%)",
        [("avg(cpu_pct)", "AVG(`cpu_pct`)", "average across replicas")],
        6, 20, percent_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_memory", "Memory Usage (%)",
        [("avg(mem_pct)", "AVG(`mem_pct`)", "average across replicas")],
        0, 26, percent_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_concurrency", "Provisioned Concurrency",
        [
            (
                "avg(provisioned_concurrency)",
                "AVG(`provisioned_concurrency`)",
                "provisioned concurrency",
            )
        ],
        6, 26,
    ),
]
pages = [
    {
        "name": "overview",
        "displayName": "Overview",
        "pageType": "PAGE_TYPE_CANVAS",
        "layoutVersion": "GRID_V1",
        "layout": overview_layout,
    }
]

if inference_query:
    datasets.append(
        {
            "name": "ds_inference",
            "displayName": "Inference table request metrics",
            "queryLines": _query_lines(inference_query),
        }
    )
    pages.append(
        {
            "name": "inference",
            "displayName": "Inference tables",
            "pageType": "PAGE_TYPE_CANVAS",
            "layoutVersion": "GRID_V1",
            "layout": [
                _text_widget("inference_title", "## Inference table metrics", 0, 0, 12, 1),
                _text_widget(
                    "inference_subtitle",
                    "Request-level aggregates from inference tables discovered through the endpoint API.",
                    0, 1, 12, 1,
                ),
                _filter_widget(
                    "ds_inference", "endpoint_name", "Model endpoint", "inference_filter_endpoint",
                    "filter-single-select", 0, 2, 4,
                ),
                _filter_widget(
                    "ds_inference", "observed_at", "Time range", "inference_filter_time",
                    "filter-date-range-picker", 4, 2, 8,
                ),
                _counter_widget(
                    "ds_inference", "observed_at", "inference_requests", "Inference requests",
                    "sum(request_count)", "SUM(`request_count`)", 0, 4, count_format,
                ),
                _counter_widget(
                    "ds_inference", "observed_at", "inference_errors", "Inference errors",
                    "sum(error_count)", "SUM(`error_count`)", 3, 4, count_format,
                ),
                _counter_widget(
                    "ds_inference", "observed_at", "inference_latency_p99", "Inference latency p99 (ms)",
                    "avg(latency_p99)", "AVG(`latency_p99`)", 6, 4, ms_format,
                ),
                _counter_widget(
                    "ds_inference", "observed_at", "inference_ttfb_p99", "Time to first byte p99 (ms)",
                    "avg(time_to_first_byte_p99)", "AVG(`time_to_first_byte_p99`)", 9, 4, ms_format,
                ),
                _line_widget(
                    "ds_inference", "observed_at", "inference_latency", "Inference latency (ms)",
                    [
                        ("avg(latency_p50)", "AVG(`latency_p50`)", "p50"),
                        ("avg(latency_p99)", "AVG(`latency_p99`)", "p99"),
                    ],
                    0, 8, ms_format,
                ),
                _line_widget(
                    "ds_inference", "observed_at", "inference_volume", "Inference request volume",
                    [
                        ("sum(request_count)", "SUM(`request_count`)", "requests"),
                        ("sum(error_count)", "SUM(`error_count`)", "errors"),
                    ],
                    6, 8, count_format,
                ),
            ],
        }
    )

dashboard_definition = {
    "datasets": datasets,
    "pages": pages,
    "uiSettings": {
        "theme": {
            "canvasBackgroundColor": {"light": "#FCFCFC", "dark": "#1F272D"},
            "widgetBackgroundColor": {"light": "#FFFFFF", "dark": "#11171C"},
            "fontColor": {"light": "#11171C", "dark": "#E8ECF0"},
            "selectionColor": {"light": "#2272B4", "dark": "#8ACAFF"},
            "visualizationColors": [
                "#003F5C", "#594E90", "#BC4C96", "#FF5F66",
                "#FFA600", "#4E5185", "#99DDB4",
            ],
            "widgetHeaderAlignment": "LEFT",
        }
    },
}

## Publish the dashboard

Create or update the AI/BI dashboard, publish its latest revision, and report links to the configured resources.

In [ ]:
dashboard_list_response = workspace.api_client.do(
    "GET",
    "/api/2.0/lakeview/dashboards",
)
matching_dashboards = [
    dashboard
    for dashboard in dashboard_list_response.get("dashboards", [])
    if dashboard.get("display_name") == DASHBOARD_NAME
    and dashboard.get("lifecycle_state") != "TRASHED"
]
if len(matching_dashboards) > 1:
    raise RuntimeError(
        f"Multiple active dashboards named {DASHBOARD_NAME!r} exist: "
        f"{[dashboard.get('dashboard_id') for dashboard in matching_dashboards]}"
    )
existing_dashboard = matching_dashboards[0] if matching_dashboards else None
_print_log(
    "dashboard_configuration_resolved",
    dashboard_name=DASHBOARD_NAME,
    existing_dashboard_id=(existing_dashboard or {}).get("dashboard_id"),
    warehouse_id=resolved_warehouse_id,
    parent_path=resolved_parent_path,
    dataset_names=[dataset["name"] for dataset in dashboard_definition["datasets"]],
    page_names=[page["name"] for page in dashboard_definition["pages"]],
)
dashboard_body = {
    "display_name": DASHBOARD_NAME,
    "warehouse_id": resolved_warehouse_id,
    "serialized_dashboard": json.dumps(dashboard_definition),
}
dashboard_query = {
    "dataset_catalog": STORAGE_CATALOG,
    "dataset_schema": STORAGE_SCHEMA,
}
if existing_dashboard:
    dashboard_id = existing_dashboard["dashboard_id"]
    dashboard = workspace.api_client.do(
        "PATCH",
        f"/api/2.0/lakeview/dashboards/{dashboard_id}",
        query=dashboard_query,
        body=dashboard_body,
    )
    dashboard_action = "updated"
else:
    dashboard = workspace.api_client.do(
        "POST",
        "/api/2.0/lakeview/dashboards",
        query=dashboard_query,
        body={**dashboard_body, "parent_path": resolved_parent_path},
    )
    dashboard_id = dashboard.get("dashboard_id")
    if not dashboard_id:
        raise RuntimeError(f"Dashboard create response did not contain dashboard_id: {dashboard}")
    dashboard_action = "created"

_print_log(
    "dashboard_draft_applied",
    action=dashboard_action,
    dashboard_id=dashboard_id,
)
published_dashboard = workspace.api_client.do(
    "POST",
    f"/api/2.0/lakeview/dashboards/{dashboard_id}/published",
    body={
        "embed_credentials": True,
        "warehouse_id": resolved_warehouse_id,
    },
)

_print_log(
    "setup_complete",
    include_endpoint_names=INCLUDE_ENDPOINT_NAMES,
    discovered_non_foundation_endpoints=RESOLVED_ENDPOINT_NAMES,
    configured_inference_tables=CONFIGURED_INFERENCE_TABLES,
    available_inference_tables=AVAILABLE_INFERENCE_TABLES,
    inference_table_status=INFERENCE_TABLE_STATUS,
    job={
        "action": job_action,
        "job_id": job_id,
        "mode": "continuous",
        "performance_target": COLLECTOR_PERFORMANCE_TARGET,
        "poll_interval_seconds": COLLECTION_INTERVAL_SECONDS,
        "url": f"{WORKSPACE_HOST}/jobs/{job_id}",
    },
    dashboard={
        "action": dashboard_action,
        "dashboard_id": dashboard_id,
        "published_at": published_dashboard.get("revision_create_time"),
        "url": f"{WORKSPACE_HOST}/dashboardsv3/{dashboard_id}/published",
    },
)